# Chapter 11 &mdash; Designing a CFG by Growing the Language "Inside-Out"

**Concept 7 of the Chapter 11 decomposition:** *Designing a CFG by Growing the Language "Inside-Out"*

Seed with $\varepsilon$, then wrap layers around it like an onion.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Designing-Inside-Out/Concept-Designing-Inside-Out.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A reliable design method:

1. **seed** with the shortest member &mdash; usually $\varepsilon$;
2. ask "**what may I wrap around a member to get another member?**";
3. each answer becomes one production;
4. ask "**can two members be joined?**" &mdash; if so, add a concatenation rule.

The picture is an **onion**: each production adds one layer around an existing
member. Because every rule is justified by a closure property of the intended
language, the grammar comes out **consistent** by construction, and completeness
follows if your list of wrappings is exhaustive.

Contrast this with guessing rules and testing afterwards, which usually produces a
grammar that is *almost* right.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### Design log for $L_{Dyck}$

In [ ]:
LOG = [("seed",  "'' is balanced",                    "S -> ''"),
       ("wrap",  "if x is balanced so is (x)",        "S -> (S)"),
       ("join",  "if x and y are balanced so is xy",  "S -> SS")]
for kind, why, rule in LOG:
    print("%-6s %-38s %s" % (kind, why, rule))
Dyck = mkg({'S': ["", "(S)", "SS"]})

### The same method on $\{a^nb^n\}$ and on even-length palindromes

In [ ]:
AnBn = mkg({'S': ["", "aSb"]})                 # seed '', wrap a...b -- no join!
Pal  = mkg({'S': ["", "aSa", "bSb"]})          # seed '', wrap with a pair

<!-- nav-strip -->

---

&larr;&nbsp;[Ch11&nbsp;6.&nbsp;Completeness and Consistency of a Grammar](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Completeness-And-Consistency/Concept-Completeness-And-Consistency.ipynb) &nbsp;&middot;&nbsp; [**Chapter 11** index](https://github.com/ganeshutah/Jove/blob/master/Chapter11/README.md) &nbsp;&middot;&nbsp; [Ch11&nbsp;8.&nbsp;The Hill/Valley Plot, and Proofs of Consistency and Completeness](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Hill-Valley-Plot/Concept-Hill-Valley-Plot.ipynb)&nbsp;&rarr;

---

## 3. Tests

Each production is justified by a closure property, so consistency is free.

In [ ]:
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0
L = language(Dyck, 8)
assert all(balanced(w) for w in L)
print("every generated string is balanced -- by construction, not by luck")

Completeness: is the list of wrappings exhaustive?

In [ ]:
from itertools import product
want = {''.join(p) for k in range(0, 9, 2) for p in product('()', repeat=k)
        if balanced(''.join(p))}
print("missing :", sorted(want - set(L)))
assert want == set(L)
print("no gaps -- the three rules cover every balanced string")

**Knowing when NOT to add a join rule** is half the method.

In [ ]:
print("L(a^n b^n) :", language(AnBn, 8))
Wrong = mkg({'S': ["", "aSb", "SS"]})          # a join rule that does NOT hold
print("with a join rule :", language(Wrong, 6))
assert 'abab' in language(Wrong, 6)
assert 'abab' not in language(AnBn, 8)
print("\n'abab' is not of the form a^n b^n, so the join rule is unsound here.")

Palindromes: two wrappings, no join.

In [ ]:
P = language(Pal, 6)
print("even-length palindromes :", P[:10], "...")
assert all(w == w[::-1] and len(w) % 2 == 0 for w in P)
assert set(P) == {w for k in range(0, 7, 2)
                  for w in (''.join(p) for p in product('ab', repeat=k))
                  if w == w[::-1]}

And the odd-length ones need one more seed, not one more wrapping.

In [ ]:
Pal2 = mkg({'S': ["", "a", "b", "aSa", "bSb"]})
P2 = language(Pal2, 5)
print("all palindromes :", P2[:12], "...")
assert all(w == w[::-1] for w in P2)
assert 'aba' in P2 and 'aa' in P2

## 4. Exercises


1. Design a grammar for "equal numbers of `a` and `b`" inside-out. Do you need a join rule?
2. Which step of the method gives consistency, and which gives completeness?
3. Add a second bracket kind to $L_{Dyck}$. How many new rules?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter11/Concept-Designing-Inside-Out')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')